# CelebA Joint-Control Experiment

This experiment compares Base, Leak, Cost, and Joint policies under one fixed protocol. Every policy starts from an empty knowledge state, acquires exactly 15 queries, and uses `lambda_q = 0`. The cost-only weight is selected on validation accuracy and cost. The joint weights are selected with a validation-only grid search that balances conditional-probe leakage reduction and cumulative-cost reduction while limiting the accuracy loss. The held-out test set is used only for the final five-seed comparison.

In [1]:
%load_ext autoreload
%autoreload 2

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import t
from torch.utils.data import DataLoader, Subset

from claq.analysis import (
    fit_conditional_probe,
    fixed_horizon_rollout,
    summarize_fixed_horizon,
)
from claq.config import CelebAClaqConfig, default_paths
from claq.core import (
    build_concept_dictionary,
    build_concept_qa_inputs,
    build_uncertainty_cost,
    concept_answers_batch,
    encode_images,
    file_sha256,
    load_answer_cache,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_run_bundle,
    make_cached_answer_loader,
    make_sensitive_mask,
    save_answer_cache,
    save_bundle_checkpoint,
)
from claq.data import (
    get_celeba_concept_qa_loaders,
    get_celeba_datasets,
    load_celeba_attribute_spec,
)
from claq.models import ConceptNet2
from claq.training import (
    HistorySamplingConfig,
    build_claq_models,
    fit_claq,
    fit_concept_qa,
    seed_everything,
)

In [2]:
repo_root = Path.cwd().resolve()
if not (repo_root / "claq").exists() and (repo_root.parent / "claq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()
config = CelebAClaqConfig()
device = config.device
runs_dir = paths.runs_root

EXPERIMENT = "celeba_joint"
PROTOCOL_VERSION = 1
SELECTION_VERSION = 2
QA_EXPERIMENT = "celeba_attractive"
SEEDS = (0, 1, 2, 3, 4)
SCREENING_SEED = 0
HORIZON = 15
NUM_EPOCHS = config.default_train_epochs
LAMBDA_S_LEAK = 0.4
LAMBDA_Q = 0.0
COST_LAMBDA_C_CANDIDATES = (0.01, 0.03, 0.1, 0.2, 0.3)
JOINT_LAMBDA_S_CANDIDATES = (0.4, 0.8, 1.2)
JOINT_LAMBDA_C_CANDIDATES = (0.001, 0.003, 0.01)
MAX_COST_VALIDATION_ACCURACY_DROP = 0.01
MAX_JOINT_VALIDATION_ACCURACY_DROP = 0.05
CALIBRATION_SEED = 1729
CALIBRATION_SIZE = 2_000
PROBE_TRAIN_SIZE = 15_000
PROBE_TUNING_SIZE = 5_000
PROBE_VALIDATION_SIZE = 10_000
PROBE_SUBSET_SEED = 2718
ACTOR_EPS = config.actor_eps
DOWNLOAD_CELEBA = False

seed_everything(SCREENING_SEED)
print({"device": str(device), "horizon": HORIZON, "seeds": SEEDS, "lambda_q": LAMBDA_Q})

{'device': 'cuda', 'horizon': 15, 'seeds': (0, 1, 2, 3, 4), 'lambda_q': 0.0}


In [3]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
spec = load_celeba_attribute_spec(
    root=paths.data_root,
    target_attribute=config.target_attribute,
    sensitive_attributes=config.sensitive_attributes,
    download=DOWNLOAD_CELEBA,
)
concepts = spec.concept_names
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sens_idx = spec.sensitive_indices
sensitive_mask = make_sensitive_mask(len(concepts), sens_idx, device)
sample_sensitive_attribute = "Male"
sample_sens_idx = torch.tensor(
    [spec.query_attribute_names.index(sample_sensitive_attribute)], dtype=torch.long
)

qa_train_loader, qa_validation_loader = get_celeba_concept_qa_loaders(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    download=DOWNLOAD_CELEBA,
)
train_dataset, validation_dataset, test_dataset = get_celeba_datasets(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    return_query_targets=True,
    download=DOWNLOAD_CELEBA,
)
if CALIBRATION_SIZE >= len(train_dataset):
    raise ValueError("CALIBRATION_SIZE must be smaller than the training split")

split_generator = torch.Generator().manual_seed(CALIBRATION_SEED)
permutation = torch.randperm(len(train_dataset), generator=split_generator).tolist()
calibration_indices = permutation[:CALIBRATION_SIZE]
policy_train_indices = permutation[CALIBRATION_SIZE:]
calibration_loader = DataLoader(
    Subset(train_dataset, calibration_indices),
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
)

print({
    "queries": len(concepts),
    "policy_train": len(policy_train_indices),
    "calibration": len(calibration_indices),
    "validation": len(validation_dataset),
    "test": len(test_dataset),
})

{'queries': 39, 'policy_train': 160770, 'calibration': 2000, 'validation': 19867, 'test': 19962}


In [4]:
qa_checkpoint = paths.checkpoints_root / f"concept_qa_{QA_EXPERIMENT}.pt"
if qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(qa_checkpoint, device=device)
else:
    qa_model = ConceptNet2().to(device)
    qa_optimizer = torch.optim.Adam(qa_model.parameters(), lr=config.learning_rate)
    qa_history = fit_concept_qa(
        model=qa_model,
        train_loader=qa_train_loader,
        eval_loader=qa_validation_loader,
        optimizer=qa_optimizer,
        scheduler=None,
        num_epochs=config.concept_qa_epochs,
        model_clip=model_clip,
        dictionary=dictionary,
        class_concept_targets=None,
        clip_device=device,
        train_device=device,
    )
    torch.save(qa_model.state_dict(), qa_checkpoint)
    with open(runs_dir / f"concept_qa_{QA_EXPERIMENT}_history.json", "w", encoding="utf-8") as handle:
        json.dump(qa_history, handle, indent=2)
    answering_model = qa_model.eval()

cache_dir = paths.artifacts_root / "concept_answers" / QA_EXPERIMENT
cache_metadata = {
    "dataset": "celeba",
    "target_attribute": spec.target_attribute,
    "concept_count": len(concepts),
    "qa_checkpoint": qa_checkpoint.name,
    "qa_checkpoint_sha256": file_sha256(qa_checkpoint),
    "threshold": config.threshold_for_binarization,
    "sensitive_target": sample_sensitive_attribute,
}
cache_paths = {
    "train": cache_dir / "train_hard_answers.pt",
    "validation": cache_dir / "validation_hard_answers.pt",
    "test": cache_dir / "test_hard_answers.pt",
}

@torch.no_grad()
def build_answer_cache(dataset, path):
    loader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=device.type == "cuda",
    )
    answer_parts, label_parts, sensitive_parts = [], [], []
    for images, labels, concept_targets in loader:
        answer_parts.append(concept_answers_batch(
            images=images,
            model_clip=model_clip,
            dictionary=dictionary,
            answering_model=answering_model,
            clip_device=device,
            train_device=device,
            threshold=config.threshold_for_binarization,
        ).cpu())
        label_parts.append(labels.cpu())
        sensitive_parts.append(concept_targets[:, sample_sens_idx].amax(dim=1).float().cpu())
    save_answer_cache(
        path,
        answers=torch.cat(answer_parts),
        labels=torch.cat(label_parts),
        sensitive_targets=torch.cat(sensitive_parts),
        metadata=cache_metadata,
    )

for split, dataset in {
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset,
}.items():
    if not cache_paths[split].exists():
        build_answer_cache(dataset, cache_paths[split])

answer_caches = {
    split: load_answer_cache(path, expected_metadata=cache_metadata)
    for split, path in cache_paths.items()
}
policy_train_cache = {
    key: value[policy_train_indices]
    for key, value in answer_caches["train"].items()
    if key in {"answers", "labels", "sensitive_targets"}
}
policy_train_loader = make_cached_answer_loader(
    policy_train_cache,
    batch_size=config.batch_size,
    shuffle=True,
    pin_memory=device.type == "cuda",
)
validation_loader = make_cached_answer_loader(
    answer_caches["validation"],
    batch_size=config.batch_size,
    shuffle=False,
    pin_memory=device.type == "cuda",
)
print({split: tuple(cache["answers"].shape) for split, cache in answer_caches.items()})

{'train': (162770, 39), 'validation': (19867, 39), 'test': (19962, 39)}


In [5]:
@torch.no_grad()
def soft_response_probabilities(loader, qa_chunk=4096):
    answering_model.eval()
    probabilities = []
    for images, _labels, _targets in loader:
        image_features = encode_images(model_clip=model_clip, images=images, device=device)
        qa_inputs = build_concept_qa_inputs(
            image_features=image_features,
            dictionary=dictionary,
        ).to(next(answering_model.parameters()).device).float()
        logits = []
        for start in range(0, len(qa_inputs), qa_chunk):
            logits.append(answering_model(qa_inputs[start : start + qa_chunk]))
        probabilities.append(
            torch.sigmoid(torch.cat(logits).view(images.size(0), len(concepts))).cpu()
        )
    return torch.cat(probabilities)

calibration_soft = soft_response_probabilities(calibration_loader)
uncertainty_cost = build_uncertainty_cost(
    calibration_soft,
    learned_mask=torch.ones(len(concepts)),
    device=device,
)
COST_VECTOR_SHA256 = hashlib.sha256(
    uncertainty_cost.detach().cpu().numpy().tobytes()
).hexdigest()
cost_table = pd.DataFrame({
    "query_index": np.arange(len(concepts)),
    "attribute": spec.query_attribute_names,
    "concept": concepts,
    "gender_associated": sensitive_mask.cpu().numpy().astype(bool),
    "uncertainty_cost": uncertainty_cost.cpu().numpy(),
})
cost_table.to_csv(runs_dir / f"{EXPERIMENT}_cost_vector.csv", index=False)
display(cost_table.sort_values("uncertainty_cost", ascending=False).head(10))

,query_index,attribute,concept,gender_associated,uncertainty_cost
24,24,Oval_Face,oval face,False,1.894180
26,26,Pointy_Nose,pointy nose,False,1.866068
5,5,Big_Lips,big lips,False,1.847866
31,31,Straight_Hair,straight hair,False,1.802759
2,2,Bags_Under_Eyes,bags under eyes,False,1.783754
6,6,Big_Nose,big nose,False,1.774442
1,1,Arched_Eyebrows,arched eyebrows,False,1.771436
10,10,Brown_Hair,brown hair,False,1.768258
36,36,Wearing_Necklace,wearing necklace,False,1.749686
32,32,Wavy_Hair,wavy hair,False,1.734621


In [6]:
def load_or_train_policy(run_name, lambda_s, lambda_c, seed):
    seed_everything(seed)
    stem = f"{EXPERIMENT}_{run_name}_seed_{seed}"
    checkpoint_path = paths.checkpoints_root / f"{stem}_best.pt"
    history_path = runs_dir / f"{stem}_history.csv"

    if checkpoint_path.exists() and history_path.exists():
        bundle = load_run_bundle(
            checkpoint_path,
            device=device,
            max_queries=len(concepts),
            num_classes=config.num_classes,
            actor_eps=ACTOR_EPS,
        )
        meta = bundle["meta"]
        expected = {
            "protocol_version": PROTOCOL_VERSION,
            "run_name": run_name,
            "seed": seed,
            "horizon": HORIZON,
            "lambda_s": float(lambda_s),
            "lambda_c": float(lambda_c),
            "lambda_q": LAMBDA_Q,
            "actor_eps": ACTOR_EPS,
            "cost_vector_sha256": COST_VECTOR_SHA256,
            "sensitive_conditioning": "conditional_y",
        }
        if any(meta.get(key) != value for key, value in expected.items()):
            raise ValueError(f"Checkpoint protocol mismatch for {checkpoint_path.name}")
        bundle.update({
            "run_name": run_name,
            "seed": seed,
            "lambda_s": lambda_s,
            "lambda_c": lambda_c,
            "history": pd.read_csv(history_path),
            "best_epoch": int(meta["best_epoch"]),
        })
        return bundle

    actor, classifier, s_head = build_claq_models(
        max_queries=len(concepts),
        num_classes=config.num_classes,
        device=device,
        actor_eps=ACTOR_EPS,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=config.learning_rate,
    )
    history_rows, best = fit_claq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=policy_train_loader,
        test_loader=validation_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=HistorySamplingConfig(0, 0, non_sensitive_only=False),
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_s=lambda_s,
        lambda_c=lambda_c,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=NUM_EPOCHS,
        sensitive_target_mode="max",
        sensitive_target_indices=sample_sens_idx,
        cost_vector=uncertainty_cost,
        training_rollout_steps=HORIZON,
        lambda_q=LAMBDA_Q,
        designated_query_mask=sensitive_mask,
    )
    history = pd.DataFrame(history_rows).assign(run_name=run_name, seed=seed)
    history.to_csv(history_path, index=False)
    actor.load_state_dict(best["actor_state_dict"])
    classifier.load_state_dict(best["classifier_state_dict"])
    s_head.load_state_dict(best["s_head_state_dict"])
    save_bundle_checkpoint(
        checkpoint_path,
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        metadata={
            "experiment": EXPERIMENT,
            "protocol_version": PROTOCOL_VERSION,
            "run_name": run_name,
            "seed": seed,
            "horizon": HORIZON,
            "lambda_s": float(lambda_s),
            "lambda_c": float(lambda_c),
            "lambda_q": LAMBDA_Q,
            "actor_eps": ACTOR_EPS,
            "cost_vector_sha256": COST_VECTOR_SHA256,
            "sensitive_conditioning": "conditional_y",
            "best_epoch": int(best["epoch"]),
            "best_validation_accuracy": float(best["test_acc"]),
        },
    )
    bundle = load_run_bundle(
        checkpoint_path,
        device=device,
        max_queries=len(concepts),
        num_classes=config.num_classes,
        actor_eps=ACTOR_EPS,
    )
    bundle.update({
        "run_name": run_name,
        "seed": seed,
        "lambda_s": lambda_s,
        "lambda_c": lambda_c,
        "history": history,
        "best_epoch": int(best["epoch"]),
    })
    return bundle

In [7]:
validation_answers = answer_caches["validation"]["answers"].float()
validation_labels = answer_caches["validation"]["labels"].long()
validation_sensitive = answer_caches["validation"]["sensitive_targets"].long()

probe_generator = torch.Generator().manual_seed(PROBE_SUBSET_SEED)
train_probe_positions = torch.randperm(
    len(policy_train_indices), generator=probe_generator
)[: PROBE_TRAIN_SIZE + PROBE_TUNING_SIZE]
validation_probe_positions = torch.randperm(
    len(validation_answers), generator=probe_generator
)[: min(PROBE_VALIDATION_SIZE, len(validation_answers))]
probe_train_positions = train_probe_positions[:PROBE_TRAIN_SIZE]
probe_tuning_positions = train_probe_positions[
    PROBE_TRAIN_SIZE : PROBE_TRAIN_SIZE + PROBE_TUNING_SIZE
]

probe_train = {
    "answers": policy_train_cache["answers"][probe_train_positions].float(),
    "labels": policy_train_cache["labels"][probe_train_positions].long(),
    "sensitive": policy_train_cache["sensitive_targets"][probe_train_positions].long(),
}
probe_tuning = {
    "answers": policy_train_cache["answers"][probe_tuning_positions].float(),
    "labels": policy_train_cache["labels"][probe_tuning_positions].long(),
    "sensitive": policy_train_cache["sensitive_targets"][probe_tuning_positions].long(),
}
probe_validation = {
    "answers": validation_answers[validation_probe_positions],
    "labels": validation_labels[validation_probe_positions],
    "sensitive": validation_sensitive[validation_probe_positions],
}


def rollout_cached(run, data):
    return fixed_horizon_rollout(
        actor=run["actor"],
        classifier=run["classifier"],
        answers=data["answers"],
        labels=data["labels"],
        sensitive_targets=data["sensitive"],
        cost_vector=uncertainty_cost,
        sensitive_mask=sensitive_mask,
        horizon=HORIZON,
        device=device,
        batch_size=config.batch_size,
    )


base_screen = load_or_train_policy("base", lambda_s=0.0, lambda_c=0.0, seed=SCREENING_SEED)
cost_screening_runs = {0.0: base_screen}
for lambda_c in COST_LAMBDA_C_CANDIDATES:
    name = f"screen_c_{lambda_c:g}".replace(".", "p")
    cost_screening_runs[lambda_c] = load_or_train_policy(
        name, lambda_s=0.0, lambda_c=lambda_c, seed=SCREENING_SEED
    )

cost_screening_rows = []
for lambda_c, run in cost_screening_runs.items():
    rollout = fixed_horizon_rollout(
        actor=run["actor"],
        classifier=run["classifier"],
        answers=validation_answers,
        labels=validation_labels,
        sensitive_targets=validation_sensitive,
        cost_vector=uncertainty_cost,
        sensitive_mask=sensitive_mask,
        horizon=HORIZON,
        device=device,
        batch_size=config.batch_size,
    )
    metrics = summarize_fixed_horizon(rollout, horizon=HORIZON, include_macro_f1=False)
    cost_screening_rows.append({"lambda_c": lambda_c, **metrics})

lambda_c_screen = pd.DataFrame(cost_screening_rows).sort_values("lambda_c").reset_index(drop=True)
base_accuracy = float(lambda_c_screen.loc[lambda_c_screen["lambda_c"].eq(0), "accuracy"].iloc[0])
cost_eligible = lambda_c_screen[
    lambda_c_screen["lambda_c"].gt(0)
    & lambda_c_screen["accuracy"].ge(base_accuracy - MAX_COST_VALIDATION_ACCURACY_DROP)
]
if cost_eligible.empty:
    cost_selected = lambda_c_screen[lambda_c_screen["lambda_c"].gt(0)].sort_values(
        ["accuracy", "mean_cumulative_cost"], ascending=[False, True]
    ).iloc[0]
else:
    cost_selected = cost_eligible.sort_values(
        ["mean_cumulative_cost", "accuracy"], ascending=[True, False]
    ).iloc[0]
LAMBDA_C_COST = float(cost_selected["lambda_c"])
lambda_c_screen.to_csv(runs_dir / f"{EXPERIMENT}_lambda_c_screen.csv", index=False)

base_train_rollout = rollout_cached(base_screen, probe_train)
base_tuning_rollout = rollout_cached(base_screen, probe_tuning)
base_validation_rollout = rollout_cached(base_screen, probe_validation)
base_probe = fit_conditional_probe(
    train_states=base_train_rollout["knowledge_states"],
    train_labels=base_train_rollout["labels"],
    train_sensitive=base_train_rollout["sensitive_targets"],
    validation_states=base_tuning_rollout["knowledge_states"],
    validation_labels=base_tuning_rollout["labels"],
    validation_sensitive=base_tuning_rollout["sensitive_targets"],
    test_states=base_validation_rollout["knowledge_states"],
    test_labels=base_validation_rollout["labels"],
    test_sensitive=base_validation_rollout["sensitive_targets"],
    num_classes=config.num_classes,
    random_state=SCREENING_SEED,
)
base_validation_metrics = summarize_fixed_horizon(
    base_validation_rollout, horizon=HORIZON, include_macro_f1=False
)

joint_screening_runs = {}
joint_screening_rows = [{
    "lambda_s": 0.0,
    "lambda_c": 0.0,
    **base_validation_metrics,
    "probe_leakage_bits": base_probe["probe_leakage_bits"],
}]
for lambda_s in JOINT_LAMBDA_S_CANDIDATES:
    for lambda_c in JOINT_LAMBDA_C_CANDIDATES:
        name = f"screen_joint_s_{lambda_s:g}_c_{lambda_c:g}".replace(".", "p")
        run = load_or_train_policy(
            name, lambda_s=lambda_s, lambda_c=lambda_c, seed=SCREENING_SEED
        )
        joint_screening_runs[(lambda_s, lambda_c)] = run
        train_rollout = rollout_cached(run, probe_train)
        tuning_rollout = rollout_cached(run, probe_tuning)
        validation_rollout = rollout_cached(run, probe_validation)
        validation_metrics = summarize_fixed_horizon(
            validation_rollout, horizon=HORIZON, include_macro_f1=False
        )
        probe = fit_conditional_probe(
            train_states=train_rollout["knowledge_states"],
            train_labels=train_rollout["labels"],
            train_sensitive=train_rollout["sensitive_targets"],
            validation_states=tuning_rollout["knowledge_states"],
            validation_labels=tuning_rollout["labels"],
            validation_sensitive=tuning_rollout["sensitive_targets"],
            test_states=validation_rollout["knowledge_states"],
            test_labels=validation_rollout["labels"],
            test_sensitive=validation_rollout["sensitive_targets"],
            num_classes=config.num_classes,
            random_state=SCREENING_SEED,
        )
        joint_screening_rows.append({
            "lambda_s": lambda_s,
            "lambda_c": lambda_c,
            **validation_metrics,
            "probe_leakage_bits": probe["probe_leakage_bits"],
        })

joint_screen = pd.DataFrame(joint_screening_rows).sort_values(
    ["lambda_s", "lambda_c"]
).reset_index(drop=True)
base_joint_row = joint_screen[
    joint_screen["lambda_s"].eq(0) & joint_screen["lambda_c"].eq(0)
].iloc[0]
joint_screen["leakage_reduction_fraction"] = (
    base_joint_row["probe_leakage_bits"] - joint_screen["probe_leakage_bits"]
) / base_joint_row["probe_leakage_bits"]
joint_screen["cost_reduction_fraction"] = (
    base_joint_row["mean_cumulative_cost"] - joint_screen["mean_cumulative_cost"]
) / base_joint_row["mean_cumulative_cost"]
joint_screen["simultaneous_control_score"] = joint_screen[
    ["leakage_reduction_fraction", "cost_reduction_fraction"]
].min(axis=1)

joint_candidates = joint_screen[joint_screen["lambda_s"].gt(0)].copy()
accuracy_eligible = joint_candidates[
    joint_candidates["accuracy"].ge(
        base_joint_row["accuracy"] - MAX_JOINT_VALIDATION_ACCURACY_DROP
    )
]
if not accuracy_eligible.empty:
    joint_candidates = accuracy_eligible
joint_improving = joint_candidates[
    joint_candidates["leakage_reduction_fraction"].gt(0)
    & joint_candidates["cost_reduction_fraction"].gt(0)
]
if not joint_improving.empty:
    joint_candidates = joint_improving
joint_selected = joint_candidates.sort_values(
    ["simultaneous_control_score", "accuracy"], ascending=[False, False]
).iloc[0]
LAMBDA_S_JOINT = float(joint_selected["lambda_s"])
LAMBDA_C_JOINT = float(joint_selected["lambda_c"])

joint_screen.to_csv(runs_dir / f"{EXPERIMENT}_joint_screen.csv", index=False)
print({
    "lambda_c_cost": LAMBDA_C_COST,
    "lambda_s_joint": LAMBDA_S_JOINT,
    "lambda_c_joint": LAMBDA_C_JOINT,
})
display(lambda_c_screen)
display(joint_screen)

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Selected lambda_c=0.2


,lambda_c,accuracy,mean_cumulative_cost,sensitive_query_rate
0,0.00,0.749534,23.759432,0.217993
1,0.01,0.742588,22.642181,0.399419
2,0.03,0.743998,22.543344,0.402560
3,0.10,0.743696,22.542313,0.400003
4,0.20,0.743847,22.540846,0.400000
5,0.30,0.743847,22.540846,0.400000


In [8]:
configurations = {
    "base": {"lambda_s": 0.0, "lambda_c": 0.0},
    "leak": {"lambda_s": LAMBDA_S_LEAK, "lambda_c": 0.0},
    "cost": {"lambda_s": 0.0, "lambda_c": LAMBDA_C_COST},
    "joint": {"lambda_s": LAMBDA_S_JOINT, "lambda_c": LAMBDA_C_JOINT},
}

runs = {}
for seed in SEEDS:
    runs[seed] = {}
    for run_name, weights in configurations.items():
        if seed == SCREENING_SEED and run_name == "base":
            run = base_screen
        elif seed == SCREENING_SEED and run_name == "joint":
            run = joint_screening_runs[(LAMBDA_S_JOINT, LAMBDA_C_JOINT)]
        else:
            run = load_or_train_policy(run_name, seed=seed, **weights)
        runs[seed][run_name] = run

print({seed: list(seed_runs) for seed, seed_runs in runs.items()})

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

...

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

{0: ['base', 'leak', 'cost', 'joint'], 1: ['base', 'leak', 'cost', 'joint'], 2: ['base', 'leak', 'cost', 'joint'], 3: ['base', 'leak', 'cost', 'joint'], 4: ['base', 'leak', 'cost', 'joint']}


In [9]:
test_data = {
    "answers": answer_caches["test"]["answers"].float(),
    "labels": answer_caches["test"]["labels"].long(),
    "sensitive": answer_caches["test"]["sensitive_targets"].long(),
}

rows = []
for seed, seed_runs in runs.items():
    for run_name, run in seed_runs.items():
        train_rollout = rollout_cached(run, probe_train)
        tuning_rollout = rollout_cached(run, probe_tuning)
        test_rollout = rollout_cached(run, test_data)
        test_metrics = summarize_fixed_horizon(
            test_rollout, horizon=HORIZON, include_macro_f1=False
        )
        probe = fit_conditional_probe(
            train_states=train_rollout["knowledge_states"],
            train_labels=train_rollout["labels"],
            train_sensitive=train_rollout["sensitive_targets"],
            validation_states=tuning_rollout["knowledge_states"],
            validation_labels=tuning_rollout["labels"],
            validation_sensitive=tuning_rollout["sensitive_targets"],
            test_states=test_rollout["knowledge_states"],
            test_labels=test_rollout["labels"],
            test_sensitive=test_rollout["sensitive_targets"],
            num_classes=config.num_classes,
            random_state=seed,
        )
        rows.append({
            "dataset": "CelebA",
            "run_name": run_name,
            "seed": seed,
            "horizon": HORIZON,
            **configurations[run_name],
            **test_metrics,
            "probe_leakage_bits": probe["probe_leakage_bits"],
            "probe_test_cross_entropy_bits": probe["test_cross_entropy_bits"],
            "probe_conditional_entropy_bits": probe["conditional_entropy_bits"],
            "probe_selected_c": probe["selected_c"],
            "probe_test_accuracy": probe["test_probe_accuracy"],
        })

results_by_seed = pd.DataFrame(rows)
display(results_by_seed)

/opt/conda/envs/claq/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/conda/envs/claq/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/conda/envs/claq/lib/python3.12/site-packa

,dataset,run_name,seed,horizon,lambda_s,lambda_c,accuracy,mean_cumulative_cost,sensitive_query_rate,probe_leakage_bits,probe_test_cross_entropy_bits,probe_conditional_entropy_bits,probe_selected_c,probe_test_accuracy
0,CelebA,base,0,15,0.0,0.0,0.769863,23.760708,0.217800,0.781187,0.065735,0.846922,10.0,0.990732
1,CelebA,leak,0,15,0.4,0.0,0.714658,24.124647,0.000000,0.152241,0.694681,0.846922,10.0,0.757990
2,CelebA,cost,0,15,0.0,0.2,0.759142,22.540850,0.400000,0.784143,0.062779,0.846922,10.0,0.990432
3,CelebA,joint,0,15,0.4,0.2,0.759393,22.540846,0.400000,0.784140,0.062782,0.846922,10.0,0.990432
4,CelebA,base,1,15,0.0,0.0,0.780333,24.380247,0.192118,0.777169,0.069753,0.846922,10.0,0.988979
5,CelebA,leak,1,15,0.4,0.0,0.719768,24.572428,0.000775,0.142243,0.704679,0.846922,10.0,0.736499
6,CelebA,cost,1,15,0.0,0.2,0.758591,22.540846,0.400000,0.784140,0.062782,0.846922,10.0,0.990432
7,CelebA,joint,1,15,0.4,0.2,0.739405,22.540846,0.400010,0.784140,0.062782,0.846922,10.0,0.990432
8,CelebA,base,2,15,0.0,0.0,0.773419,24.067513,0.258421,0.783056,0.063866,0.846922,10.0,0.990532
9,CelebA,leak,2,15,0.4,0.0,0.702735,24.382959,0.066667,0.208590,0.638332,0.846922,0.1,0.764903


In [10]:
metrics = [
    "accuracy",
    "probe_leakage_bits",
    "mean_cumulative_cost",
    "sensitive_query_rate",
]
t_critical = float(t.ppf(0.975, len(SEEDS) - 1))
summary_rows = []
for run_name in configurations:
    group = results_by_seed[results_by_seed["run_name"].eq(run_name)]
    row = {"dataset": "CelebA", "run_name": run_name, "seeds": group["seed"].nunique()}
    for metric in metrics:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_ci95"] = t_critical * group[metric].std(ddof=1) / np.sqrt(len(group))
    summary_rows.append(row)
results_summary = pd.DataFrame(summary_rows)

results_by_seed.to_csv(runs_dir / f"{EXPERIMENT}_results_by_seed.csv", index=False)
results_summary.to_csv(runs_dir / f"{EXPERIMENT}_results_summary.csv", index=False)
manifest = {
    "experiment": EXPERIMENT,
    "protocol_version": PROTOCOL_VERSION,
    "selection_version": SELECTION_VERSION,
    "horizon": HORIZON,
    "lambda_q": LAMBDA_Q,
    "lambda_s_leak": LAMBDA_S_LEAK,
    "lambda_c_cost": LAMBDA_C_COST,
    "lambda_s_joint": LAMBDA_S_JOINT,
    "lambda_c_joint": LAMBDA_C_JOINT,
    "probe_train_size": len(probe_train["answers"]),
    "probe_tuning_size": len(probe_tuning["answers"]),
    "probe_validation_size": len(probe_validation["answers"]),
    "joint_screen_path": str(
        (runs_dir / f"{EXPERIMENT}_joint_screen.csv").relative_to(repo_root)
    ),
    "checkpoints": {
        f"{run_name}_seed_{seed}": str(run["ckpt_path"].relative_to(repo_root))
        for seed, seed_runs in runs.items()
        for run_name, run in seed_runs.items()
    },
}
with open(paths.checkpoints_root / f"{EXPERIMENT}_checkpoints.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

display(results_summary)

,dataset,run_name,seeds,accuracy_mean,accuracy_ci95,probe_leakage_bits_mean,probe_leakage_bits_ci95,mean_cumulative_cost_mean,mean_cumulative_cost_ci95,sensitive_query_rate_mean,sensitive_query_rate_ci95
0,CelebA,base,5,0.772949,0.005693,0.696933,0.148066,24.306623,0.496679,0.222747,0.045299
1,CelebA,leak,5,0.731640,0.033660,0.251981,0.147435,24.387079,0.212956,0.053466,0.068979
2,CelebA,cost,5,0.758080,0.002187,0.784141,0.000002,22.540847,0.000002,0.400000,0.000000
3,CelebA,joint,5,0.747260,0.009873,0.705253,0.134350,22.573463,0.055581,0.395978,0.011439
